<a href="https://colab.research.google.com/github/AnaMartins2022/diariodaalegria/blob/main/Maia_Jan2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import folium
import requests
import json
from shapely.geometry import shape, Point

print("Setup concluído.")

Ambiente preparado para 11 áreas de influência (incluindo o Centro Urbano).


In [73]:
# 1. Definição do Limite Oficial (Data fornecida por ti)
concelho_data = {
    "type": "Polygon",
    "coordinates": [[
        [-8.621777, 41.295369], [-8.62117, 41.295124], [-8.620466, 41.295153], [-8.619983, 41.294668],
        [-8.619774, 41.294503], [-8.619453, 41.294391], [-8.618044, 41.2941], [-8.616635, 41.293809],
        [-8.616299, 41.293697], [-8.615146, 41.293094], [-8.614013, 41.292468], [-8.612922, 41.291895],
        [-8.609829, 41.207351], [-8.631412, 41.225706], [-8.65259, 41.23846], [-8.66601, 41.260959],
        [-8.621777, 41.295369] # Ponto de fecho
    ]]
}
mascara_maia = shape(concelho_data)

# 2. Isócrona Lidador (Exemplo de entrada)
lidador_raw = Polygon([[-8.66601, 41.260959], [-8.671022, 41.259962], [-8.670597, 41.250363], [-8.661729, 41.25695], [-8.66601, 41.260959]])

# 3. Operação de Recorte (Intersection)
# Isto garante que a mancha só aparece DENTRO do concelho
lidador_final = lidador_raw.intersection(mascara_maia)

print("✅ Clipping geográfico aplicado com sucesso.")

✅ Clipping geográfico aplicado com sucesso.


In [77]:
def buscar_servicos_validos(poligono_recortado):
    # Categorias: townhall (Câmara/Juntas), school, university, pharmacy, bank, hospital
    amenities = ['townhall', 'university', 'school', 'pharmacy', 'bank', 'hospital']
    bbox = "41.18, -8.72, 41.32, -8.55"
    query = f"[out:json]; ({''.join([f'node[\"amenity\"=\"{a}\"]({bbox});way[\"amenity\"=\"{a}\"]({bbox});' for a in amenities])}); out center;"

    data = requests.get("https://overpass-api.de/api/interpreter", params={'data': query}).json()

    estilos = {
        'townhall': {'cor': 'darkblue', 'ico': 'building-columns', 'label': 'Câmara/Junta'},
        'school': {'cor': 'blue', 'ico': 'book', 'label': 'Ensino'},
        'pharmacy': {'cor': 'red', 'ico': 'plus-medical', 'label': 'Saúde'}
    }

    filtrados = []
    for item in data.get('elements', []):
        lat = item.get('lat') or item.get('center', {}).get('lat')
        lon = item.get('lon') or item.get('center', {}).get('lon')
        ponto = Point(lon, lat)

        # SÓ ADICIONA SE ESTIVER DENTRO DA ISÓCRONA RECORTADA
        if poligono_recortado.contains(ponto):
            tags = item.get('tags', {})
            tipo = tags.get('amenity')
            est = estilos.get(tipo, {'cor': 'gray', 'ico': 'info', 'label': 'Serviço'})
            filtrados.append({
                'nome': tags.get('name', est['label']),
                'coords': [lat, lon],
                'cor': est['cor'],
                'icone': est['ico']
            })
    return filtrados

pontos_lidador = buscar_servicos_validos(lidador_final)
print(f"📍 Encontrados {len(pontos_lidador)} serviços no Lidador.")

📍 Encontrados 0 serviços no Lidador.


In [78]:
# Criar o mapa
m = folium.Map(location=[41.255, -8.666], zoom_start=14, tiles="OpenStreetMap")
Fullscreen().add_to(m)

# 1. Desenhar o seu limite fornecido (Linha PRETA grossa)
folium.GeoJson(
    limite_data,
    name="Limite do Concelho da Maia",
    style_function=lambda x: {
        'color': 'black',
        'weight': 6,
        'fillOpacity': 0
    }
).add_to(m)

# 2. Desenhar a Isócrona Recortada (Verde)
folium.GeoJson(
    lidador_recortado,
    name="Área 10m Lidador",
    style_function=lambda x: {
        'fillColor': '#2ecc71',
        'color': '#27ae60',
        'weight': 3,
        'fillOpacity': 0.3
    }
).add_to(m)

# 3. Marcador da Estação
folium.Marker(
    [41.25495, -8.66801],
    popup="Estação Metro Lidador",
    icon=folium.Icon(color='purple', icon='train', prefix='fa')
).add_to(m)

# 4. Marcadores de Serviços
for s in servicos_lidador:
    folium.Marker(
        s['coords'],
        popup=f"<b>{s['nome']}</b><br>Tipo: {s['tipo']}",
        icon=folium.Icon(color='blue', icon='info-sign')
    ).add_to(m)

folium.LayerControl().add_to(m)
m

NameError: name 'limite_data' is not defined